# 

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:

import torch

print("GPU Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))


In [ ]:

import os

# Display available datasets in Kaggle
dataset_path = "/kaggle/input"

print("Available datasets:")
for dataset in os.listdir(dataset_path):
    print("📁", dataset)


In [ ]:

import os

# Find dataset files and folders
for root, dirs, files in os.walk("/kaggle/input/datasets"):
    print("Folder:", root)
    print("Subfolders:", dirs)
    print("Files:", files[:10])
    print("-" * 50)

    # Limit output to avoid too much text
    if root.count("/") >= 6:
        dirs[:] = []


In [ ]:

import os
import yaml

# Exact Kaggle dataset location
DATASET_PATH = "/kaggle/input/datasets/sadiiamjad/helmet-detection-v6-balanced-dataset-yolov8"

# Locate data.yaml
yaml_path = os.path.join(DATASET_PATH, "data.yaml")

# Read configuration
with open(yaml_path, "r") as file:
    data = yaml.safe_load(file)

print("DATASET CONFIGURATION")
print("=" * 40)

print("Number of classes:", data.get("nc"))
print("Class names:", data.get("names"))

print("\nTraining path:", data.get("train"))
print("Validation path:", data.get("val"))
print("Testing path:", data.get("test"))

print("\nComplete YAML configuration:")
print(data)


In [ ]:

import os
import matplotlib.pyplot as plt

# Dataset path from previous step
DATASET_PATH = "/kaggle/input/datasets/sadiiamjad/helmet-detection-v6-balanced-dataset-yolov8"

# Count images and label files
splits = ["train", "valid", "test"]
image_counts = []
label_counts = []

for split in splits:
    image_folder = os.path.join(DATASET_PATH, split, "images")
    label_folder = os.path.join(DATASET_PATH, split, "labels")

    images = [
        f for f in os.listdir(image_folder)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    labels = [
        f for f in os.listdir(label_folder)
        if f.endswith(".txt")
    ]

    image_counts.append(len(images))
    label_counts.append(len(labels))

    print(f"{split.upper()}")
    print(f"Images: {len(images)}")
    print(f"Label files: {len(labels)}")
    print("-" * 30)

print("Total images:", sum(image_counts))

# Visualise the dataset distribution
plt.figure(figsize=(8, 5))
bars = plt.bar(
    ["Training", "Validation", "Testing"],
    image_counts,
    color=["royalblue", "seagreen", "orange"]
)

plt.title("Helmet Detection Dataset Distribution")
plt.xlabel("Dataset Split")
plt.ylabel("Number of Images")

for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height + 10,
        str(int(height)),
        ha="center"
    )

plt.ylim(0, max(image_counts) * 1.15)
plt.tight_layout()
plt.savefig(
    "/kaggle/working/dataset_distribution.png",
    dpi=300
)
plt.show()


In [ ]:

import os
from collections import Counter
import matplotlib.pyplot as plt

DATASET_PATH = "/kaggle/input/datasets/sadiiamjad/helmet-detection-v6-balanced-dataset-yolov8"

class_names = ["With Helmet", "Without Helmet", "licence"]
splits = ["train", "valid", "test"]

class_counts = {}

for split in splits:
    label_folder = os.path.join(DATASET_PATH, split, "labels")
    counts = Counter()

    for filename in os.listdir(label_folder):
        if filename.endswith(".txt"):
            filepath = os.path.join(label_folder, filename)

            with open(filepath, "r") as file:
                for line in file:
                    values = line.strip().split()

                    if len(values) == 5:
                        class_id = int(values[0])
                        if 0 <= class_id < len(class_names):
                            counts[class_id] += 1

    class_counts[split] = counts

    print(f"\n{split.upper()} CLASS DISTRIBUTION")

    for class_id, name in enumerate(class_names):
        print(f"{name}: {counts[class_id]}")

# Visualise distribution across splits
x = range(len(class_names))
width = 0.25

plt.figure(figsize=(10, 6))

for i, split in enumerate(splits):
    values = [
        class_counts[split][class_id]
        for class_id in range(len(class_names))
    ]

    positions = [p + i * width for p in x]

    plt.bar(
        positions,
        values,
        width=width,
        label=split.capitalize()
    )

plt.xticks(
    [p + width for p in x],
    class_names
)

plt.xlabel("Object Classes")
plt.ylabel("Number of Annotations")
plt.title("Helmet Detection Class Distribution")
plt.legend()
plt.tight_layout()

plt.savefig(
    "/kaggle/working/class_distribution.png",
    dpi=300
)

plt.show()


In [ ]:

import os
import random
import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

DATASET_PATH = "/kaggle/input/datasets/sadiiamjad/helmet-detection-v6-balanced-dataset-yolov8"

image_dir = os.path.join(DATASET_PATH, "train", "images")
label_dir = os.path.join(DATASET_PATH, "train", "labels")

class_names = ["With Helmet", "Without Helmet", "licence"]
colors = ["green", "red", "blue"]

# Select 6 random training images
random.seed(42)
image_files = [
    f for f in os.listdir(image_dir)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]
selected_images = random.sample(
    sorted(image_files),
    min(6, len(image_files))
)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for ax, image_file in zip(axes.flatten(), selected_images):
    image_path = os.path.join(image_dir, image_file)
    label_path = os.path.join(
        label_dir,
        os.path.splitext(image_file)[0] + ".txt"
    )

    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    height, width = image.shape[:2]

    ax.imshow(image)

    if os.path.exists(label_path):
        with open(label_path, "r") as file:
            for line in file:
                values = line.strip().split()

                if len(values) != 5:
                    continue

                class_id = int(values[0])
                x_center, y_center, box_w, box_h = map(
                    float, values[1:]
                )

                # Convert YOLO normalised coordinates
                # into pixel coordinates
                x = (x_center - box_w / 2) * width
                y = (y_center - box_h / 2) * height
                w = box_w * width
                h = box_h * height

                rect = Rectangle(
                    (x, y), w, h,
                    linewidth=2,
                    edgecolor=colors[class_id],
                    facecolor="none"
                )

                ax.add_patch(rect)
                ax.text(
                    x, max(y - 5, 0),
                    class_names[class_id],
                    color=colors[class_id],
                    fontsize=9,
                    fontweight="bold",
                    bbox=dict(facecolor="white", alpha=0.8)
                )

    ax.set_title(image_file[:35], fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.savefig(
    "/kaggle/working/annotated_samples.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


In [ ]:

import os
import shutil

# Original dataset
SOURCE_PATH = "/kaggle/input/datasets/sadiiamjad/helmet-detection-v6-balanced-dataset-yolov8"

# Working dataset location
WORKING_PATH = "/kaggle/working/helmet_dataset"

# Copy the dataset only if needed
if not os.path.exists(WORKING_PATH):
    shutil.copytree(SOURCE_PATH, WORKING_PATH)
    print("Dataset copied successfully!")
else:
    print("Working dataset already exists.")

# Verify the copied folders
for split in ["train", "valid", "test"]:
    image_dir = os.path.join(WORKING_PATH, split, "images")
    label_dir = os.path.join(WORKING_PATH, split, "labels")

    print(f"\n{split.upper()}")
    print("Images:", len(os.listdir(image_dir)))
    print("Label files:", len(os.listdir(label_dir)))

print("\nWorking dataset location:", WORKING_PATH)


In [ ]:

import os
import hashlib
from collections import defaultdict

WORKING_PATH = "/kaggle/working/helmet_dataset"
splits = ["train", "valid", "test"]

missing_labels = []
empty_labels = []
invalid_annotations = []
image_hashes = defaultdict(list)

for split in splits:
    image_dir = os.path.join(WORKING_PATH, split, "images")
    label_dir = os.path.join(WORKING_PATH, split, "labels")

    for image_name in sorted(os.listdir(image_dir)):
        if not image_name.lower().endswith(
            (".jpg", ".jpeg", ".png")
        ):
            continue

        image_path = os.path.join(image_dir, image_name)
        label_name = os.path.splitext(image_name)[0] + ".txt"
        label_path = os.path.join(label_dir, label_name)

        # Find duplicate images using SHA256
        with open(image_path, "rb") as f:
            image_hash = hashlib.sha256(f.read()).hexdigest()

        image_hashes[image_hash].append((split, image_name))

        if not os.path.exists(label_path):
            missing_labels.append((split, image_name))
            continue

        with open(label_path, "r") as f:
            lines = f.readlines()

        if not any(line.strip() for line in lines):
            empty_labels.append((split, image_name))

        for line_number, line in enumerate(lines, start=1):
            values = line.strip().split()

            if not values:
                continue

            try:
                if len(values) != 5:
                    raise ValueError("Incorrect number of values")

                class_value = float(values[0])
                if not class_value.is_integer():
                    raise ValueError("Non-integer class ID")

                class_id = int(class_value)
                x, y, w, h = map(float, values[1:])

                if class_id not in [0, 1, 2]:
                    raise ValueError("Invalid class ID")

                if not (
                    0 <= x <= 1 and
                    0 <= y <= 1 and
                    0 < w <= 1 and
                    0 < h <= 1 and
                    x - w / 2 >= -1e-6 and
                    x + w / 2 <= 1 + 1e-6 and
                    y - h / 2 >= -1e-6 and
                    y + h / 2 <= 1 + 1e-6
                ):
                    raise ValueError("Invalid bounding box")

            except ValueError as error:
                invalid_annotations.append(
                    (split, image_name, line_number, str(error))
                )

# Only report duplicates across different splits
cross_split_duplicates = [
    locations
    for locations in image_hashes.values()
    if len({split for split, name in locations}) > 1
]

print("DATA QUALITY VALIDATION REPORT")
print("=" * 45)

print("Missing labels:", len(missing_labels))
print("Empty labels:", len(empty_labels))
print("Invalid annotations:", len(invalid_annotations))
print("Cross-split duplicate groups:",
      len(cross_split_duplicates))

print("\nEmpty label examples:")
print(empty_labels[:10])

print("\nInvalid annotation examples:")
print(invalid_annotations[:10])

print("\nCross-split duplicate examples:")
for group in cross_split_duplicates[:10]:
    print(group)


In [ ]:

import os
import math
import csv

WORKING_PATH = "/kaggle/working/helmet_dataset"
splits = ["train", "valid", "test"]

corrections = []
unresolved = []

for split in splits:
    label_dir = os.path.join(WORKING_PATH, split, "labels")

    for filename in sorted(os.listdir(label_dir)):
        if not filename.endswith(".txt"):
            continue

        filepath = os.path.join(label_dir, filename)

        with open(filepath, "r") as file:
            lines = file.readlines()

        updated_lines = []
        file_changed = False

        for line_number, line in enumerate(lines, 1):
            values = line.strip().split()

            if not values:
                updated_lines.append(line)
                continue

            try:
                if len(values) != 5:
                    raise ValueError("Unexpected number of values")

                class_id = int(values[0])
                x, y, w, h = map(float, values[1:])

                if class_id not in [0, 1, 2]:
                    raise ValueError("Invalid class ID")

                if not all(math.isfinite(v) for v in [x, y, w, h]):
                    raise ValueError("Non-finite coordinates")

                if w <= 0 or h <= 0:
                    raise ValueError("Non-positive box dimensions")

                # Convert centre-width-height to corners
                x1 = x - w / 2
                y1 = y - h / 2
                x2 = x + w / 2
                y2 = y + h / 2

                # Clip corners to image boundaries
                x1 = max(0.0, min(1.0, x1))
                y1 = max(0.0, min(1.0, y1))
                x2 = max(0.0, min(1.0, x2))
                y2 = max(0.0, min(1.0, y2))

                if x2 <= x1 or y2 <= y1:
                    raise ValueError("Box has no area after clipping")

                new_x = (x1 + x2) / 2
                new_y = (y1 + y2) / 2
                new_w = x2 - x1
                new_h = y2 - y1

                new_line = (
                    f"{class_id} {new_x:.8f} {new_y:.8f} "
                    f"{new_w:.8f} {new_h:.8f}\n"
                )

                if new_line.strip() != line.strip():
                    file_changed = True
                    corrections.append(
                        [split, filename, line_number,
                         line.strip(), new_line.strip()]
                    )

                updated_lines.append(new_line)

            except (ValueError, OverflowError) as error:
                unresolved.append(
                    [split, filename, line_number, str(error)]
                )
                updated_lines.append(line)

        if file_changed:
            with open(filepath, "w") as file:
                file.writelines(updated_lines)

# Save an audit trail of corrections
log_path = "/kaggle/working/bbox_corrections.csv"

with open(log_path, "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow([
        "split", "file", "line",
        "original", "corrected"
    ])
    writer.writerows(corrections)

print("BOUNDING BOX CORRECTION REPORT")
print("=" * 45)
print("Annotations rewritten:", len(corrections))
print("Unresolved annotations:", len(unresolved))
print("Correction log:", log_path)

if unresolved:
    print("\nUnresolved examples:")
    for item in unresolved[:10]:
        print(item)


In [ ]:

import os
import hashlib
from collections import defaultdict

WORKING_PATH = "/kaggle/working/helmet_dataset"

# Find exact duplicate images across dataset splits
image_hashes = defaultdict(list)

for split in ["train", "valid", "test"]:
    image_dir = os.path.join(WORKING_PATH, split, "images")

    for filename in sorted(os.listdir(image_dir)):
        if not filename.lower().endswith(
            (".jpg", ".jpeg", ".png")
        ):
            continue

        image_path = os.path.join(image_dir, filename)

        with open(image_path, "rb") as f:
            image_hash = hashlib.sha256(f.read()).hexdigest()

        image_hashes[image_hash].append((split, filename))

# Prefer retaining training images
priority = {"train": 0, "valid": 1, "test": 2}
removed = []

for locations in image_hashes.values():
    if len({split for split, name in locations}) < 2:
        continue

    locations = sorted(
        locations,
        key=lambda x: (priority[x[0]], x[1])
    )

    # Keep the first image and remove copies in other splits
    for split, filename in locations[1:]:
        image_path = os.path.join(
            WORKING_PATH, split, "images", filename
        )

        label_path = os.path.join(
            WORKING_PATH, split, "labels",
            os.path.splitext(filename)[0] + ".txt"
        )

        if os.path.exists(image_path):
            os.remove(image_path)

        if os.path.exists(label_path):
            os.remove(label_path)

        removed.append((split, filename))

print("DUPLICATE REMOVAL REPORT")
print("=" * 45)
print("Duplicate images removed:", len(removed))

for item in removed:
    print("Removed:", item)

print("\nUPDATED DATASET COUNTS")

for split in ["train", "valid", "test"]:
    image_dir = os.path.join(WORKING_PATH, split, "images")

    count = len([
        f for f in os.listdir(image_dir)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ])

    print(f"{split.upper()}: {count} images")


In [ ]:

import os
import math
import hashlib
from collections import defaultdict

WORKING_PATH = "/kaggle/working/helmet_dataset"

splits = ["train", "valid", "test"]
hashes = defaultdict(list)

missing_labels = []
empty_labels = []
invalid_boxes = []
total_boxes = 0

for split in splits:
    image_dir = os.path.join(WORKING_PATH, split, "images")
    label_dir = os.path.join(WORKING_PATH, split, "labels")

    images = sorted([
        f for f in os.listdir(image_dir)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ])

    for image_name in images:
        image_path = os.path.join(image_dir, image_name)

        # Check for exact duplicate images
        with open(image_path, "rb") as f:
            image_hash = hashlib.sha256(f.read()).hexdigest()

        hashes[image_hash].append((split, image_name))

        # Locate corresponding label
        label_name = os.path.splitext(image_name)[0] + ".txt"
        label_path = os.path.join(label_dir, label_name)

        if not os.path.exists(label_path):
            missing_labels.append((split, image_name))
            continue

        with open(label_path, "r") as f:
            lines = [line.strip() for line in f if line.strip()]

        if not lines:
            empty_labels.append((split, image_name))

        for line_number, line in enumerate(lines, 1):
            try:
                values = line.split()

                if len(values) != 5:
                    raise ValueError("Incorrect label format")

                class_value = float(values[0])

                if not math.isfinite(class_value):
                    raise ValueError("Non-finite class ID")

                if not class_value.is_integer():
                    raise ValueError("Non-integer class ID")

                class_id = int(class_value)
                x, y, w, h = map(float, values[1:])

                if class_id not in (0, 1, 2):
                    raise ValueError("Invalid class")

                if not all(math.isfinite(v) for v in (x, y, w, h)):
                    raise ValueError("Non-finite coordinates")

                if not (
                    0 <= x <= 1 and
                    0 <= y <= 1 and
                    0 < w <= 1 and
                    0 < h <= 1 and
                    x - w / 2 >= -1e-6 and
                    x + w / 2 <= 1 + 1e-6 and
                    y - h / 2 >= -1e-6 and
                    y + h / 2 <= 1 + 1e-6
                ):
                    raise ValueError("Invalid bounding box")

                total_boxes += 1

            except (ValueError, OverflowError) as error:
                invalid_boxes.append(
                    (split, image_name, line_number, str(error))
                )

cross_split_duplicates = [
    locations for locations in hashes.values()
    if len({split for split, _ in locations}) > 1
]

print("FINAL DATASET VALIDATION")
print("=" * 40)
print("Total images:", sum(
    len([
        f for f in os.listdir(
            os.path.join(WORKING_PATH, split, "images")
        )
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ])
    for split in splits
))
print("Valid bounding boxes:", total_boxes)
print("Missing labels:", len(missing_labels))
print("Empty labels:", len(empty_labels))
print("Invalid annotations:", len(invalid_boxes))
print("Cross-split duplicate groups:", len(cross_split_duplicates))

print("\nInvalid annotation examples:")
print(invalid_boxes[:10])


In [ ]:

import os
import yaml

# Cleaned dataset location
WORKING_PATH = "/kaggle/working/helmet_dataset"

# Correct YOLOv8 configuration
dataset_config = {
    "path": WORKING_PATH,
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 3,
    "names": [
        "With Helmet",
        "Without Helmet",
        "licence"
    ]
}

# Save configuration
CONFIG_PATH = "/kaggle/working/helmet_data.yaml"

with open(CONFIG_PATH, "w") as file:
    yaml.safe_dump(
        dataset_config,
        file,
        sort_keys=False
    )

# Verify configuration
with open(CONFIG_PATH, "r") as file:
    print(file.read())

# Confirm dataset folders
for split in ["train", "val", "test"]:
    folder = os.path.join(
        WORKING_PATH,
        dataset_config[split]
    )

    assert os.path.isdir(folder), f"Missing folder: {folder}"
    print(f"{split.upper()} folder verified!")

print("\nYOLOv8 dataset configuration ready!")


In [ ]:

%pip install -q ultralytics


In [ ]:

import torch
import ultralytics

print("DEEP LEARNING ENVIRONMENT CHECK")
print("=" * 45)

print("PyTorch version:", torch.__version__)
print("Ultralytics version:", ultralytics.__version__)

print("\nGPU INFORMATION")
print("=" * 45)

if torch.cuda.is_available():
    print("CUDA available: YES")
    print("GPU Name:", torch.cuda.get_device_name(0))
    print("Number of GPUs:", torch.cuda.device_count())
    print("CUDA Version:", torch.version.cuda)
else:
    print("CUDA available: NO")
    print("Please enable a GPU in Kaggle settings.")

print("\nEnvironment verification completed!")


In [ ]:

from ultralytics import YOLO
import os

# Location of the cleaned dataset configuration
CONFIG_PATH = "/kaggle/working/helmet_data.yaml"

# Verify dataset configuration exists
assert os.path.isfile(CONFIG_PATH), "Dataset YAML file not found!"

# Load pretrained YOLOv8 Nano model
model = YOLO("yolov8n.pt")

print("=" * 45)
print("HELMET DETECTION BASELINE MODEL")
print("=" * 45)

print("Model: YOLOv8 Nano")
print("Pretrained weights: yolov8n.pt")
print("Dataset configuration:", CONFIG_PATH)
print("Number of target classes: 3")

print("\nModel loaded successfully!")

# Display the neural network architecture
model.info()


In [ ]:

from ultralytics import YOLO
import torch

# Confirm GPU availability
assert torch.cuda.is_available(), "GPU is not available!"

# Load pretrained YOLOv8 Nano
model = YOLO("yolov8n.pt")

# Start baseline model training
results = model.train(
    data="/kaggle/working/helmet_data.yaml",
    epochs=30,
    imgsz=640,
    batch=16,
    device=0,
    optimizer="AdamW",
    lr0=0.001,
    patience=10,

    # Data augmentation
    degrees=10.0,
    translate=0.1,
    scale=0.3,
    fliplr=0.5,
    mosaic=1.0,

    # Reproducibility
    seed=42,
    deterministic=True,

    # Save training results
    project="/kaggle/working/helmet_runs",
    name="yolov8n_baseline",
    exist_ok=True,
    plots=True,
    save=True
)

print("BASELINE TRAINING COMPLETED!")
print("Results saved at:")
print("/kaggle/working/helmet_runs/yolov8n_baseline")


In [ ]:

from ultralytics import YOLO
import os

# Load the best trained helmet detection model
BEST_MODEL_PATH = (
    "/kaggle/working/helmet_runs/"
    "yolov8n_baseline/weights/best.pt"
)

assert os.path.exists(BEST_MODEL_PATH), "Trained model not found!"

best_model = YOLO(BEST_MODEL_PATH)

# Evaluate on the untouched test dataset
test_results = best_model.val(
    data="/kaggle/working/helmet_data.yaml",
    split="test",
    imgsz=640,
    batch=16,
    device=0,
    plots=True,
    project="/kaggle/working/helmet_runs",
    name="baseline_test_evaluation",
    exist_ok=True
)

print("\nFINAL TEST EVALUATION")
print("=" * 45)

print(f"Precision: {test_results.box.mp:.4f}")
print(f"Recall: {test_results.box.mr:.4f}")
print(f"mAP@50: {test_results.box.map50:.4f}")
print(f"mAP@50-95: {test_results.box.map:.4f}")

print("\nResults saved to:")
print("/kaggle/working/helmet_runs/baseline_test_evaluation")


In [ ]:

from ultralytics import YOLO
import os
import random
import cv2
import matplotlib.pyplot as plt

# Load our trained model
model = YOLO(
    "/kaggle/working/helmet_runs/"
    "yolov8n_baseline/weights/best.pt"
)

# Load unseen test images
test_dir = "/kaggle/working/helmet_dataset/test/images"

image_files = sorted([
    f for f in os.listdir(test_dir)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
])

# Select six reproducible test images
random.seed(42)
selected = random.sample(image_files, min(6, len(image_files)))

# Perform inference
predictions = model.predict(
    source=[os.path.join(test_dir, f) for f in selected],
    imgsz=640,
    conf=0.25,
    device=0,
    verbose=False
)

# Display results with bounding boxes and confidence
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for ax, prediction, filename in zip(
    axes.flatten(), predictions, selected
):
    annotated = prediction.plot()
    annotated = cv2.cvtColor(
        annotated, cv2.COLOR_BGR2RGB
    )

    ax.imshow(annotated)
    ax.set_title(filename[:35], fontsize=9)
    ax.axis("off")

plt.tight_layout()

plt.savefig(
    "/kaggle/working/helmet_predictions.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Prediction demonstration completed!")
print("Saved: /kaggle/working/helmet_predictions.png")


In [ ]:

import os
import matplotlib.pyplot as plt
from PIL import Image

RUN_PATH = "/kaggle/working/helmet_runs/yolov8n_baseline"

graph_files = [
    "results.png",
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "BoxPR_curve.png"
]

for filename in graph_files:
    filepath = os.path.join(RUN_PATH, filename)

    if os.path.exists(filepath):
        print("Displaying:", filename)

        image = Image.open(filepath)

        plt.figure(figsize=(14, 9))
        plt.imshow(image)
        plt.axis("off")
        plt.title(filename)
        plt.show()
    else:
        print("Not found:", filename)


In [ ]:

from PIL import Image
import matplotlib.pyplot as plt
import os

# Use the test evaluation confusion matrix
matrix_path = (
    "/kaggle/working/helmet_runs/"
    "baseline_test_evaluation/confusion_matrix.png"
)

assert os.path.exists(matrix_path), "Confusion matrix not found!"

image = Image.open(matrix_path)

plt.figure(figsize=(10, 9))
plt.imshow(image)
plt.title("Helmet Detection — Test Confusion Matrix")
plt.axis("off")
plt.tight_layout()
plt.show()


In [ ]:

from ultralytics import YOLO
import os
from PIL import Image
import matplotlib.pyplot as plt

model = YOLO(
    "/kaggle/working/helmet_runs/"
    "yolov8n_baseline/weights/best.pt"
)

# Validate with an explicit confidence threshold
filtered_results = model.val(
    data="/kaggle/working/helmet_data.yaml",
    split="test",
    imgsz=640,
    conf=0.25,
    iou=0.7,
    device=0,
    plots=True,
    project="/kaggle/working/helmet_runs",
    name="test_conf025",
    exist_ok=True
)

# Display the new confusion matrix
matrix_path = (
    "/kaggle/working/helmet_runs/"
    "test_conf025/confusion_matrix.png"
)

assert os.path.exists(matrix_path), "Matrix not found!"

plt.figure(figsize=(10, 9))
plt.imshow(Image.open(matrix_path))
plt.axis("off")
plt.title("Test Confusion Matrix — Confidence 0.25")
plt.show()

print("Confidence-filtered evaluation completed.")


In [ ]:

import os
import zipfile

BASE = "/kaggle/working"
RUN = f"{BASE}/helmet_runs/yolov8n_baseline"
TEST = f"{BASE}/helmet_runs/baseline_test_evaluation"

files_to_save = [
    (f"{RUN}/weights/best.pt", "model/best.pt"),
    (f"{RUN}/results.csv", "training/results.csv"),
    (f"{RUN}/results.png", "training/results.png"),
    (f"{RUN}/confusion_matrix.png",
     "training/confusion_matrix.png"),
    (f"{TEST}/confusion_matrix.png",
     "evaluation/test_confusion_matrix.png"),
    (f"{BASE}/helmet_runs/test_conf025/confusion_matrix.png",
     "evaluation/test_confusion_matrix_conf025.png"),
    (f"{BASE}/helmet_predictions.png",
     "prototype/helmet_predictions.png"),
    (f"{BASE}/dataset_distribution.png",
     "dataset/dataset_distribution.png"),
    (f"{BASE}/class_distribution.png",
     "dataset/class_distribution.png"),
    (f"{BASE}/annotated_samples.png",
     "dataset/annotated_samples.png"),
    (f"{BASE}/helmet_data.yaml", "configuration/helmet_data.yaml"),
    (f"{BASE}/bbox_corrections.csv",
     "preprocessing/bbox_corrections.csv")
]

zip_path = f"{BASE}/Helmet_Detection_Milestone_2.zip"

with zipfile.ZipFile(
    zip_path, "w", zipfile.ZIP_DEFLATED
) as archive:
    for source, destination in files_to_save:
        if os.path.isfile(source):
            archive.write(source, destination)
            print("Added:", destination)
        else:
            print("Not found:", source)

print("\nEvidence package created:")
print(zip_path)
print("Size:", round(os.path.getsize(zip_path) / 1024**2, 2), "MB")
